# Latency budget

Build a p99 latency budget across components for an online ML request.
Edit the `budget` dict to model your system. The cell prints whether you fit
your SLO with a configurable headroom factor.

Rule of thumb (Huyen 2022): tail latencies do NOT add linearly. Use the
**sum of p99s** as the upper bound for the **end-to-end p99** of a serial
chain. If you have parallel fan-out, the tail is the **max** of the branches,
which is worse than any individual branch.

In [ ]:
SLO_P99_MS = 200  # the budget
HEADROOM = 0.8    # spend at most 80% of the SLO, reserve 20% for surprise

# Each entry: stage -> (p50_ms, p99_ms, parallel_group)
# Stages with the same parallel_group run concurrently; their max p99 counts.
budget = {
    'edge_tls':         (1,   4,  'A'),
    'auth_ratelimit':   (1,   3,  'A'),
    'feature_fetch':    (5,  20,  'B'),
    'candidate_recall': (8,  30,  'B'),
    'ranker':           (15, 60,  'C'),
    'post_processing':  (2,   8,  'C'),
    'response_encode':  (1,   3,  'D'),
}

from collections import defaultdict
groups = defaultdict(list)
for k, (p50, p99, g) in budget.items():
    groups[g].append((k, p50, p99))

total_p50 = sum(max(p50 for _, p50, _ in v) for v in groups.values())
total_p99 = sum(max(p99 for _, _, p99 in v) for v in groups.values())

print(f'sum p50  : {total_p50} ms')
print(f'sum p99  : {total_p99} ms  (rough upper bound on end-to-end p99)')
print(f'budget   : {SLO_P99_MS} ms  (with {int(HEADROOM*100)}% headroom => {int(SLO_P99_MS*HEADROOM)} ms)')
if total_p99 <= SLO_P99_MS * HEADROOM:
    print('FIT')
else:
    print(f'OVER by {total_p99 - SLO_P99_MS*HEADROOM:.0f} ms')
    over = sorted(budget.items(), key=lambda x: -x[1][1])[:3]
    print('top offenders:')
    for k, (p50, p99, g) in over:
        print(f'  {k:20s} p99={p99} ms')

## How to spend the budget

1. **Measure first, model second.** Replace the defaults with numbers from your traces (Tempo, Jaeger, Honeycomb).
2. **Tail dominates.** A 20 ms p99 stage with a 2 ms p50 is fine on average and fatal at p99. Optimize the tail.
3. **Parallelize what you can.** `feature_fetch` and `candidate_recall` are independent reads — run them concurrently and the budget collapses to the max of the two.
4. **Cap the model.** If the ranker blows your budget, pick a smaller one. Latency-aware model selection is cheaper than every other lever.
5. **Cache.** Prompt cache (LLM), feature cache (recsys), model cache (cold start).

See `04-serving-online-batch-streaming/README.md` for the full discussion.